# Vision Metrics

Evaluate a Vision Language Model (VLM) using two metrics based on semantic similarity between the model's description and the human ground truth.

| Metric | What it tells you |
|--------|-------------------|
| **VisionSimilarity** | How semantically close the VLM descriptions are to the ground truth on average. A score of 1.0 means identical meaning; 0.0 means completely unrelated. |
| **VisionHallucination** | How often the VLM describes something significantly different from what actually happened. A frame is a hallucination when similarity falls below `threshold`. |

In [ ]:
!pip install gaussia

Defaulting to user installation because normal site-packages is not writeable
ERROR: Ignored the following versions that require a different python version: 1.0.0 Requires-Python >=3.11; 1.1.0 Requires-Python >=3.11; 1.2.0 Requires-Python >=3.11; 1.2.0b1 Requires-Python >=3.11; 1.2.0b2 Requires-Python >=3.11; 1.2.0b3 Requires-Python >=3.11; 1.2.0b4 Requires-Python >=3.11; 1.2.0b5 Requires-Python >=3.11; 1.2.0b6 Requires-Python >=3.11; 1.2.0b7 Requires-Python >=3.11; 1.3.0b1 Requires-Python >=3.11; 2.0.0 Requires-Python >=3.11; 2.0.0b1 Requires-Python >=3.11; 2.0.0b2 Requires-Python >=3.11; 2.0.0b3 Requires-Python >=3.11; 3.0.0b1 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement gaussia (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
ERROR: No matching distribution found for gaussia


In [6]:
import logging
from gaussia.metrics.vision import VisionSimilarity, VisionHallucination

logging.getLogger('httpx').disabled = True

## Setup

Implement a `Retriever` that loads your evaluation dataset.

Each `Batch` only needs two text fields:
- `assistant` — free-text description produced by the VLM
- `ground_truth_assistant` — human-annotated description of what actually happened

In [7]:
import json
from pathlib import Path
from gaussia import Retriever
from gaussia.schemas import Dataset


class VLMRetriever(Retriever):
    def load_dataset(self) -> list[Dataset]:
        with open(Path('dataset.json'), encoding='utf-8') as f:
            return [Dataset.model_validate(item) for item in json.load(f)]


# Threshold — frames with similarity below this value are flagged as hallucinations.
# Adjust based on your domain:
#   0.85+ → strict (only near-identical descriptions pass)
#   0.75  → balanced (default)
#   0.60  → lenient (allows paraphrasing and partial descriptions)
THRESHOLD = 0.75

## Vision Similarity

How accurately does the VLM describe scenes compared to the human ground truth?

In [8]:
similarity_results = VisionSimilarity.run(VLMRetriever, threshold=THRESHOLD)

for r in similarity_results:
    r.display()

2026-03-18 11:36:58,864 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-03-18 11:36:58,865 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


cam-entrance-01:   0%|          | 0/5 [00:00<?, ?frame/s]

cam-parking-02:   0%|          | 0/4 [00:00<?, ?frame/s]

Session: cam-entrance-01  |  Assistant: argos-gpt4o
The model's descriptions have an average similarity of 62% with the ground truth (min: 17%, max: 91%) across 5 frames.

  frame_001  similarity=0.85
  frame_002  similarity=0.91
  frame_003  similarity=0.17
  frame_004  similarity=0.80
  frame_005  similarity=0.35

Session: cam-parking-02  |  Assistant: argos-gpt4o
The model's descriptions have an average similarity of 73% with the ground truth (min: 23%, max: 91%) across 4 frames.

  frame_001  similarity=0.91
  frame_002  similarity=0.90
  frame_003  similarity=0.23
  frame_004  similarity=0.86



## Vision Hallucination

How often does the VLM describe something significantly different from what actually happened?

In [9]:
hallucination_results = VisionHallucination.run(VLMRetriever, threshold=THRESHOLD)

for r in hallucination_results:
    r.display()

2026-03-18 11:37:04,328 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-03-18 11:37:04,330 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


cam-entrance-01:   0%|          | 0/5 [00:00<?, ?frame/s]

cam-parking-02:   0%|          | 0/4 [00:00<?, ?frame/s]

Session: cam-entrance-01  |  Assistant: argos-gpt4o
The model hallucinated in 2 of 5 frames (40%). A frame is considered a hallucination when similarity with the ground truth falls below 0.75.

  frame_001  similarity=0.85  ok
  frame_002  similarity=0.91  ok
  frame_003  similarity=0.17  HALLUCINATION
  frame_004  similarity=0.80  ok
  frame_005  similarity=0.35  HALLUCINATION

Session: cam-parking-02  |  Assistant: argos-gpt4o
The model hallucinated in 1 of 4 frames (25%). A frame is considered a hallucination when similarity with the ground truth falls below 0.75.

  frame_001  similarity=0.91  ok
  frame_002  similarity=0.90  ok
  frame_003  similarity=0.23  HALLUCINATION
  frame_004  similarity=0.86  ok

